<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fisica_de_Plasmas_y_Fusion/Plasmas_Espaciales_Simulacion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Simulación Computacional: Plasmas Espaciales
**Área:** Fisica de Plasmas y Fusion

Este cuaderno interactivo de Jupyter ejecuta numéricamente los modelos físicos descritos en el repositorio. Puedes modificar los parámetros iniciales y ejecutar las celdas para visualizar gráficos o animaciones interactivos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# Normalizamos variables:
# u_norm = u / c_s (Número de Mach M)
# r_norm = r / r_c
# La ecuación de Parker normalizada es: (M - 1/M) dM/dr = 2/r - 2/r^2

r_norm = np.linspace(0.1, 10, 500)

# Ecuación algebraica implícita integral del viento solar:
# M^2 - 1 - 2*ln(M) = 4*ln(r_norm) + 4/r_norm - 3
# Definimos la función para buscar raíces:
def parker_eq(M, r):
    return M**2 - 1 - 2*np.log(M) - 4*np.log(r) - 4/r + 3

M_sup = np.zeros_like(r_norm)
M_sub = np.zeros_like(r_norm)

for i, r in enumerate(r_norm):
    # Buscamos solución supersónica (M > 1)
    # y solución subsónica (M < 1)
    if r == 1.0:
        M_sup[i] = 1.0
        M_sub[i] = 1.0
    else:
        # Adivinanzas iniciales basadas en el radio
        guess_sup = r if r > 1 else 2.0
        guess_sub = 0.5/r**2 if r > 1 else 0.5
        
        sol_sup = fsolve(parker_eq, guess_sup, args=(r,))
        sol_sub = fsolve(parker_eq, guess_sub, args=(r,))
        
        M_sup[i] = sol_sup[0]
        M_sub[i] = sol_sub[0]

# Construimos la solución física completa (Viento Solar)
# Combina rama subsónica para r < 1 y supersónica para r > 1
solar_wind = np.where(r_norm <= 1.0, M_sub, M_sup)
# Brisa solar (totalmente subsónica)
solar_breeze = M_sub

plt.figure(figsize=(10, 6))
plt.plot(r_norm, solar_wind, 'r-', linewidth=3, label='Viento Solar (Transónico)')
plt.plot(r_norm, solar_breeze, 'b--', linewidth=2, label='Brisa Solar (Subsónica)')

plt.axvline(1.0, color='k', linestyle=':', label='Punto Crítico Sonico ($r_c$)')
plt.axhline(1.0, color='k', linestyle=':')

plt.title('Modelo Isotérmico del Viento Solar de Parker')
plt.xlabel('Distancia Normalizada $r / r_c$')
plt.ylabel('Velocidad Normalizada (Mach $u / c_s$)')
plt.ylim(0, 3.5)
plt.legend()
plt.grid(True)
plt.show()